# Pipeline 2 — Full DPO Training Data Pipeline

Tests the **complete DPO data flow**. DPO skips the reward model entirely — it learns directly from preference pairs.

| Step | Mode | Produces | Purpose |
|------|------|----------|---------|
| 1 | `sft` | `prompts.bin` + `responses.bin` | SFT warm-up (reference model) |
| 2 | `dpo` | `prompts.bin` + `chosen.bin` + `chosen_labels.bin` + `rejected.bin` | DPO preference learning |

> **Key difference from PPO:** No rollout phase. No reward model. One unified loss over preference pairs.

Repo: https://github.com/SniAssia/Optimized_data_loaders/tree/main/reinforcement_learning_data_loader

## 0. Config

In [ ]:
import os

REPO_URL      = "https://github.com/SniAssia/Optimized_data_loaders.git"
REPO_DIR      = "Optimized_data_loaders"
RL_DIR        = os.path.join(REPO_DIR, "reinforcement_learning_data_loader")
LIBTORCH_PATH = "/content/libtorch"

## 1. Clone / Pull Repo

In [ ]:
import subprocess

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

print("Repo contents:")
for f in sorted(os.listdir(RL_DIR)):
    print(" ", f)

## 2. LibTorch Setup

In [ ]:
import torch
print("PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda)

if not os.path.isdir(LIBTORCH_PATH):
    !wget -q https://download.pytorch.org/libtorch/nightly/cu128/libtorch-cxx11-abi-shared-with-deps-latest.zip
    !unzip -q libtorch-cxx11-abi-shared-with-deps-latest.zip
    print("libtorch ready")
else:
    print("libtorch already at", LIBTORCH_PATH)

## 3. Download Dataset

In [ ]:
jsonl_path = os.path.join(RL_DIR, "hh_rlhf_dpo.jsonl")

result = subprocess.run(
    [
        "python3", "prepare_dataset.py",
        "--dataset", "Anthropic/hh-rlhf",
        "--output",  "hh_rlhf_dpo.jsonl",
    ],
    cwd=RL_DIR, capture_output=True, text=True,
)
print("STDOUT:", result.stdout)
print("STDERR:", result.stderr[-2000:] if result.stderr else "")
result.check_returncode()

## 4. Build C++ Binary

In [ ]:
import shutil

have_cmake  = shutil.which("cmake") is not None
have_gpp    = shutil.which("g++")   is not None or shutil.which("clang++") is not None
libtorch_ok = LIBTORCH_PATH and os.path.isdir(LIBTORCH_PATH)
can_build   = have_cmake and have_gpp and libtorch_ok

build_dir = os.path.join(RL_DIR, "build")
build_ok  = False

if can_build:
    os.makedirs(build_dir, exist_ok=True)
    cfg = subprocess.run(
        ["cmake", f"-DCMAKE_PREFIX_PATH={LIBTORCH_PATH}", ".."],
        cwd=build_dir, capture_output=True, text=True,
    )
    if cfg.returncode == 0:
        bld = subprocess.run(
            ["cmake", "--build", ".", "-j"],
            cwd=build_dir, capture_output=True, text=True,
        )
        print(bld.stdout[-2000:])
        build_ok = bld.returncode == 0
    else:
        print(cfg.stderr[-1000:])

print("build_ok =", build_ok)
binary_path = os.path.join(build_dir, "dataloader_demo")

---
# DPO Phase 1 — SFT Batches (Reference Model Warm-Up)

**Why SFT first in DPO?**

DPO requires a **reference model** — a frozen copy of the policy before preference learning.
The reference model is the SFT checkpoint. Without SFT first, the DPO loss has no stable baseline.

**Batch format:** `(prompt, response, labels)` — standard next-token prediction

In [ ]:
sft_out = os.path.join(RL_DIR, "out_dpo_sft")

result = subprocess.run(
    [
        "python3", "tokenize_dataset.py",
        "--input",      "hh_rlhf_dpo.jsonl",
        "--output-dir", "out_dpo_sft",
        "--mode",       "sft",
    ],
    cwd=RL_DIR, capture_output=True, text=True,
)
print("STDOUT:", result.stdout)
print("STDERR:", result.stderr[-1500:] if result.stderr else "")
result.check_returncode()

In [ ]:
if build_ok and os.path.isfile(binary_path):
    print("========== DPO Phase 1: SFT (reference model warm-up) ==========")
    run = subprocess.run(
        [os.path.abspath(binary_path), "sft"],
        cwd=sft_out, capture_output=True, text=True,
    )
    print(run.stdout)
    if run.returncode != 0:
        print("STDERR:", run.stderr)
else:
    print("Binary not built.")

### SFT Batch Shape
```
[SFT] batch 0  prompt=[4, 128]  response=[4, 128]  labels=[4, 128]
```
After SFT training, **freeze this checkpoint** as the reference model `π_ref` for DPO.

---
# DPO Phase 2 — DPO Preference Batches

**Purpose:** Train the policy directly from preference pairs — no reward model needed.

**Batch format:** `(prompt, chosen, chosen_labels, rejected)`

**Why `chosen_labels` but not `rejected_labels`?**
- DPO loss increases log-prob of chosen responses → needs labels to know which tokens to optimize
- Rejected side only needs log-prob for contrast, not per-token supervision

**Files produced:** `prompts.bin` + `chosen.bin` + `chosen_labels.bin` + `rejected.bin`

In [ ]:
dpo_out = os.path.join(RL_DIR, "out_dpo")

result = subprocess.run(
    [
        "python3", "tokenize_dataset.py",
        "--input",      "hh_rlhf_dpo.jsonl",
        "--output-dir", "out_dpo",
        "--mode",       "dpo",
    ],
    cwd=RL_DIR, capture_output=True, text=True,
)
print("STDOUT:", result.stdout)
print("STDERR:", result.stderr[-1500:] if result.stderr else "")
result.check_returncode()

In [ ]:
for fname in ["prompts.bin", "chosen.bin", "rejected.bin"]:
    fpath = os.path.join(dpo_out, fname)
    if os.path.isfile(fpath):
        print(f"{fname}: {os.path.getsize(fpath)/1e6:.2f} MB")
    else:
        print(f"{fname}: NOT FOUND")

In [ ]:
if build_ok and os.path.isfile(binary_path):
    print("========== DPO Phase 2: DPO Preference Learning ==========")
    run = subprocess.run(
        [os.path.abspath(binary_path), "dpo"],
        cwd=dpo_out, capture_output=True, text=True,
    )
    print(run.stdout)
    if run.returncode != 0:
        print("STDERR:", run.stderr)
else:
    print("Binary not built.")

### What DPO Batches Look Like

```
[DPO] batch 0  chosen=[4, 49]  chosen_labels=[4, 49]  rejected=[4, 63]
               ↑               ↑ prompt masked -100    ↑ different length OK
```

**DPO Loss:**
```python
loss = -log(sigmoid(
    β * (log_π(chosen) - log_π_ref(chosen)) -
    β * (log_π(rejected) - log_π_ref(rejected))
))
```
Both the **active policy** and **frozen reference model** process each batch.

## Decode a Batch to Verify Token IDs Are Correct

In [ ]:
import struct
from transformers import AutoTokenizer

def read_bin(path):
    seqs = []
    with open(path, "rb") as f:
        (n,) = struct.unpack("<I", f.read(4))
        for _ in range(n):
            (length,) = struct.unpack("<H", f.read(2))
            ids = struct.unpack(f"<{length}I", f.read(4 * length))
            seqs.append(list(ids))
    return seqs

tok = AutoTokenizer.from_pretrained("inceptionai/jais-family-590m", trust_remote_code=True)

chosen   = read_bin(os.path.join(dpo_out, "chosen.bin"))
rejected = read_bin(os.path.join(dpo_out, "rejected.bin"))

idx = 0
print("=== Chosen (decoded) ===")
print(tok.decode(chosen[idx])[:300])
print()
print("=== Rejected (decoded) ===")
print(tok.decode(rejected[idx])[:300])

---
# Full DPO Pipeline Summary

```
hh_rlhf_dpo.jsonl
       │
       ├─── tokenize (sft)   → out_dpo_sft/prompts.bin + responses.bin
       │         ↓
       │    [SFT batch]  prompt=[B,T]  response=[B,T]  labels=[B,T]
       │    → trains reference model π_ref  (then FREEZE it)
       │
       └─── tokenize (dpo)   → out_dpo/prompts.bin + chosen.bin + rejected.bin
                 ↓
            [DPO batch]  chosen=[B,T]  chosen_labels=[B,T]  rejected=[B,T]
            → DPO loss updates active policy π  (comparing against frozen π_ref)
            → NO reward model needed, NO rollout phase
```

### DPO vs PPO at a Glance

| | PPO | DPO |
|--|-----|-----|
| Reward model | ✅ trained separately | ❌ not needed |
| Rollout phase | ✅ online generation | ❌ not needed |
| Batch type | single responses + scalar | preference pairs |
| Complexity | higher | lower |